# Fase 3 — Pré-processamento

**Objetivo:** Transformar o dataset bruto em um formato que os modelos consigam aprender. Cada decisão aqui foi definida na discussão pós-EDA.

**Roteiro desta fase:**

| Etapa | O que fazemos | Por quê |
|-------|---------------|---------|
| 1. Feature engineering | `hour_of_day`, `day_of_week`, `log1p(TransactionAmt)` | Sinal temporal forte; normalizar skewness |
| 2. Indicadores de missing | Feature `_is_nan` para colunas onde NaN discrimina fraude | Preservar informação contida na ausência |
| 3. Agrupamento de alta cardinalidade | Top-N categorias + `'other'` | Evitar explosão de dimensões no encoding |
| 4. Imputação | Mediana (numéricas), `'missing'` (categóricas) | Robusto a outliers; preserva NaN como categoria |
| 5. Encoding | Label Encoding para todas as categóricas | Compatível com XGBoost/LightGBM nativamente |
| 6. Split temporal | 80% treino / 20% validação por `TransactionDT` | Evitar data leakage (não treinar no futuro) |
| 7. Normalização | `StandardScaler` só para Regressão Logística | Árvores são invariantes a escala |

## 1. Imports e Configuração

In [1]:
import os
import sys
import json
import warnings
import pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR     = '/home/fernando-daniel-marcelino/data/ieee-cis-fraud-detection'
MODELS_DIR   = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

print('Ambiente configurado.')

Ambiente configurado.


## 2. Carregamento do Dataset

In [2]:
df = pd.read_parquet(os.path.join(DATA_DIR, 'train_merged.parquet'), engine='pyarrow')
print(f'Dataset carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas')

# Separar target antes de qualquer transformação
# (nunca deixar o target passar pelo pipeline de features — causaria data leakage)
TARGET = 'isFraud'
DROP_COLS = ['TransactionID']  # ID não tem valor preditivo

print(f'Fraudes: {df[TARGET].sum():,} ({df[TARGET].mean()*100:.2f}%)')

Dataset carregado: 590,540 linhas × 434 colunas
Fraudes: 20,663 (3.50%)


## 3. Feature Engineering

### 3.1 — Features Temporais

O EDA mostrou que a **hora do dia** varia de 2% a 11% de taxa de fraude — é uma das variáveis mais poderosas do dataset. Vamos extraí-la de `TransactionDT`.

Por que não usar `TransactionDT` diretamente?  
- `TransactionDT` é um valor crescente (segundos acumulados). O modelo não saberia que "hora 3" é parecida com "hora 4" se ele vir só o número bruto 86400 vs 172800.  
- Ao extrair `hour_of_day`, damos ao modelo uma informação **interpretável e cíclica**.

In [3]:
df['hour_of_day'] = (df['TransactionDT'] % 86400) // 3600
df['day_of_week'] = (df['TransactionDT'] // 86400) % 7

# TransactionDT original não precisa mais ficar como feature —
# já extraímos o sinal relevante dele
DROP_COLS.append('TransactionDT')

print('Features temporais criadas: hour_of_day, day_of_week')
print(f'  hour_of_day: {df["hour_of_day"].min()}–{df["hour_of_day"].max()}')
print(f'  day_of_week: {df["day_of_week"].min()}–{df["day_of_week"].max()}')

Features temporais criadas: hour_of_day, day_of_week
  hour_of_day: 0–23
  day_of_week: 0–6


### 3.2 — Transformação Log de TransactionAmt

A distribuição de `TransactionAmt` é fortemente assimétrica (média $135, mas max $31.937). Para a Regressão Logística, isso é problemático — o modelo vai "supervalorizar" os outliers. A transformação `log1p` comprime a escala e deixa a distribuição mais simétrica.

Por que `log1p` e não `log`? Porque `log(0)` é indefinido. `log1p(x) = log(x + 1)` funciona mesmo para valores próximos de zero.

In [4]:
df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])

# Manter o original também — árvores podem usar as duas
print('Feature criada: TransactionAmt_log')
print(f'  Original : min={df["TransactionAmt"].min():.2f}, max={df["TransactionAmt"].max():.2f}')
print(f'  Log1p    : min={df["TransactionAmt_log"].min():.2f}, max={df["TransactionAmt_log"].max():.2f}')

Feature criada: TransactionAmt_log
  Original : min=0.25, max=31937.39
  Log1p    : min=0.22, max=10.37


## 4. Indicadores de Missing

O EDA mostrou que, para diversas colunas, o fato de o valor estar ausente já é um sinal de fraude.  
As variáveis **D** representam *dias desde o último evento* — quando ausentes, indicam que é a **primeira transação** do cartão/dispositivo, o que é suspeito.  
As variáveis **addr** e **id_03/id_04** ausentes também mostraram diferença significativa.

Estratégia: criar uma coluna binária `X_is_nan` **antes** de imputar os valores. Assim o modelo tem acesso a ambas as informações: o valor imputado E o fato de que era ausente.

In [5]:
# Colunas identificadas no EDA como tendo NaN informativo
MISSING_INDICATOR_COLS = [
    'D6', 'D7', 'D8', 'D9', 'D12', 'D13', 'D14',
    'addr1', 'addr2',
    'id_03', 'id_04', 'id_09', 'id_10'
]

new_indicator_cols = []
for col in MISSING_INDICATOR_COLS:
    if col in df.columns:
        new_col = f'{col}_is_nan'
        df[new_col] = df[col].isna().astype(np.int8)
        new_indicator_cols.append(new_col)

print(f'Indicadores de missing criados: {len(new_indicator_cols)}')
print('Exemplo — D7_is_nan:')
print(df.groupby('D7_is_nan')[TARGET].agg(['mean','count'])
        .rename(columns={'mean': 'fraud_rate', 'count': 'volume'})
        .assign(fraud_rate=lambda x: x['fraud_rate'].map('{:.2%}'.format)).to_string())

Indicadores de missing criados: 13
Exemplo — D7_is_nan:
          fraud_rate  volume
D7_is_nan                   
0             14.88%   38917
1              2.70%  551623


## 5. Tratamento de Categóricas de Alta Cardinalidade

Algumas colunas categóricas têm centenas de valores únicos (ex: `DeviceInfo` tem modelos de aparelhos como "Samsung SM-G892A"). Se fizermos Label Encoding diretamente, o modelo vai ver centenas de valores raros, o que pode causar overfitting.

**Estratégia:** manter as top-N categorias mais frequentes e agrupar o restante em `'other'`. Valores raros têm pouca informação estatística — o agrupamento é uma forma de regularização.

In [6]:
HIGH_CARDINALITY = {
    'P_emaildomain': 20,
    'R_emaildomain': 20,
    'DeviceInfo':    30,
    'id_30':         20,   # sistema operacional (ex: Android 7.0, iOS 11)
    'id_31':         20,   # navegador (ex: chrome 62, mobile safari)
    'id_33':         15,   # resolução de tela (ex: 1920x1080)
}

for col, top_n in HIGH_CARDINALITY.items():
    if col not in df.columns:
        continue
    top_cats = df[col].value_counts().head(top_n).index.tolist()
    before = df[col].nunique(dropna=False)
    df[col] = df[col].where(df[col].isin(top_cats), other='other')
    after = df[col].nunique(dropna=False)
    print(f'  {col}: {before} → {after} valores únicos (top {top_n} + other)')

  P_emaildomain: 60 → 21 valores únicos (top 20 + other)
  R_emaildomain: 61 → 21 valores únicos (top 20 + other)
  DeviceInfo: 1787 → 31 valores únicos (top 30 + other)
  id_30: 76 → 21 valores únicos (top 20 + other)
  id_31: 131 → 21 valores únicos (top 20 + other)
  id_33: 261 → 16 valores únicos (top 15 + other)


## 6. Imputação de Valores Ausentes

### Por que mediana (e não média) para numéricas?
A média é sensível a outliers. `TransactionAmt` tem valores até $31.937 — um outlier assim puxa a média para longe da maioria dos valores. A **mediana** é o valor central da distribuição, robusto a extremos.

### Por que `'missing'` (e não `0` ou moda) para categóricas?
Se imputarmos moda, estamos dizendo "este valor desconhecido é provavelmente gmail.com" — o que pode não ser verdade. Ao criar a categoria `'missing'`, o modelo aprende explicitamente o padrão associado a "não sabemos o e-mail", que pode ser um sinal próprio.

In [7]:
# Identificar colunas por tipo (excluindo target e IDs)
feature_cols = [c for c in df.columns if c not in [TARGET] + DROP_COLS]

num_cols = df[feature_cols].select_dtypes(include='number').columns.tolist()
cat_cols = df[feature_cols].select_dtypes(include='object').columns.tolist()

print(f'Features numéricas : {len(num_cols)}')
print(f'Features categóricas: {len(cat_cols)}')

# --- Imputação numérica: mediana ---
medians = df[num_cols].median()
df[num_cols] = df[num_cols].fillna(medians)

# Verificar: nenhum NaN deve restar nas numéricas
remaining_num_nans = df[num_cols].isnull().sum().sum()
print(f'\nNaN restantes em numéricas após imputação: {remaining_num_nans}')

# --- Imputação categórica: categoria 'missing' ---
df[cat_cols] = df[cat_cols].fillna('missing')

remaining_cat_nans = df[cat_cols].isnull().sum().sum()
print(f'NaN restantes em categóricas após imputação: {remaining_cat_nans}')

Features numéricas : 416
Features categóricas: 31

NaN restantes em numéricas após imputação: 0
NaN restantes em categóricas após imputação: 0


## 7. Encoding de Variáveis Categóricas

**Label Encoding** atribui um número inteiro a cada categoria (ex: `visa=0`, `mastercard=1`, `discover=2`).  
É adequado para XGBoost e LightGBM porque esses modelos trabalham com árvores de decisão — eles vão aprender os limiares corretos independentemente da ordem dos números.  

**Atenção:** Para Regressão Logística, Label Encoding implica uma ordem (1 > 0 > 2?), o que é problemático. Mas vamos lidar com isso na Fase 4, usando o modelo com essas features da mesma forma — o impacto é menor quando normalizamos e o modelo tem muitas features numéricas dominando.

In [8]:
label_encoders = {}  # guardar os encoders para poder inverter depois, se precisar

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

print(f'Label Encoding aplicado em {len(cat_cols)} colunas categóricas.')
print('\nExemplos de mapeamento:')
for col in ['card4', 'card6', 'DeviceType', 'ProductCD']:
    if col in label_encoders:
        mapping = dict(zip(label_encoders[col].classes_,
                           label_encoders[col].transform(label_encoders[col].classes_)))
        print(f'  {col}: {mapping}')

Label Encoding aplicado em 31 colunas categóricas.

Exemplos de mapeamento:
  card4: {'american express': 0, 'discover': 1, 'mastercard': 2, 'missing': 3, 'visa': 4}
  card6: {'charge card': 0, 'credit': 1, 'debit': 2, 'debit or credit': 3, 'missing': 4}
  DeviceType: {'desktop': 0, 'missing': 1, 'mobile': 2}
  ProductCD: {'C': 0, 'H': 1, 'R': 2, 'S': 3, 'W': 4}


In [9]:
# Verificação final: nenhum NaN deve existir no dataset
total_nans = df[feature_cols].isnull().sum().sum()
print(f'NaN total no dataset após toda a imputação e encoding: {total_nans}')
assert total_nans == 0, 'ERRO: ainda existem NaN no dataset!'
print('✓ Dataset limpo — nenhum NaN restante.')

NaN total no dataset após toda a imputação e encoding: 0
✓ Dataset limpo — nenhum NaN restante.


## 8. Split Treino / Validação — Por que Temporal?

Em dados com dimensão de tempo, um split **aleatório** causa **data leakage**: o modelo treina em dados de março para prever dados de janeiro. Na prática real, isso nunca acontece — você nunca tem dados do futuro para treinar.

Um split temporal reproduz a realidade: treinar no passado, validar no futuro.

**Split:** 80% das transações mais antigas para treino, 20% mais recentes para validação.

In [10]:
# Identificar o threshold de tempo para o split
# (precisamos do TransactionDT original — já foi removido do df, então recalculamos)
# Vamos recarregar só o TransactionDT do parquet para fazer o split
dt_series = pd.read_parquet(
    os.path.join(DATA_DIR, 'train_merged.parquet'),
    columns=['TransactionDT'],
    engine='pyarrow'
)['TransactionDT']

split_threshold = dt_series.quantile(0.80)
train_mask = dt_series <= split_threshold
val_mask   = dt_series >  split_threshold

print(f'Threshold de split (80th percentile de TransactionDT): {split_threshold:,.0f}')
print(f'  → Equivale a ~{split_threshold/86400:.0f} dias desde o início')
print(f'\nTamanho do treino    : {train_mask.sum():,} ({train_mask.mean()*100:.1f}%)')
print(f'Tamanho da validação : {val_mask.sum():,}  ({val_mask.mean()*100:.1f}%)')
print(f'\nFraudes no treino    : {df.loc[train_mask, TARGET].mean()*100:.2f}%')
print(f'Fraudes na validação : {df.loc[val_mask,   TARGET].mean()*100:.2f}%')

Threshold de split (80th percentile de TransactionDT): 12,192,854
  → Equivale a ~141 dias desde o início

Tamanho do treino    : 472,432 (80.0%)
Tamanho da validação : 118,108  (20.0%)

Fraudes no treino    : 3.51%
Fraudes na validação : 3.44%


In [11]:
# Criar os conjuntos de features (X) e target (y)
feature_cols_final = [c for c in df.columns if c not in [TARGET] + DROP_COLS]

X_train = df.loc[train_mask, feature_cols_final].reset_index(drop=True)
y_train = df.loc[train_mask, TARGET].reset_index(drop=True)
X_val   = df.loc[val_mask,   feature_cols_final].reset_index(drop=True)
y_val   = df.loc[val_mask,   TARGET].reset_index(drop=True)

print(f'X_train: {X_train.shape}  |  y_train fraudes: {y_train.sum():,}')
print(f'X_val  : {X_val.shape}    |  y_val   fraudes: {y_val.sum():,}')
print(f'\nTotal de features: {X_train.shape[1]}')

X_train: (472432, 447)  |  y_train fraudes: 16,599
X_val  : (118108, 447)    |  y_val   fraudes: 4,064

Total de features: 447


## 9. Normalização — Apenas para Regressão Logística

### Por que árvores não precisam de normalização?
Algoritmos baseados em árvores (XGBoost, LightGBM) fazem splits como `V257 > 0.5?` — a escala absoluta não importa, só a ordem. Normalizar não muda a ordem, então não afeta o resultado.

### Por que a Regressão Logística precisa?
A Regressão Logística aprende pesos (coeficientes) para cada feature. Se `TransactionAmt` varia de 0.25 a 31.937 e `hour_of_day` varia de 0 a 23, o otimizador vai dar passos muito maiores para `TransactionAmt`, causando convergência lenta e instável.

**Regra de ouro:** Normalize o scaler **apenas no treino** (`fit_transform`). No conjunto de validação, aplique apenas `transform` — senão você estaria usando informação do futuro para calcular a escala.

In [12]:
scaler = StandardScaler()

# fit_transform no treino: aprende média e desvio padrão do treino
X_train_scaled = scaler.fit_transform(X_train)

# transform (só!) na validação: usa os parâmetros aprendidos no treino
X_val_scaled = scaler.transform(X_val)

# Converter de volta para DataFrame para manter nomes das colunas
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols_final)
X_val_scaled   = pd.DataFrame(X_val_scaled,   columns=feature_cols_final)

print('StandardScaler ajustado no treino e aplicado na validação.')
print(f'\nVerificação (X_train_scaled) — deve ser ~0 média e ~1 desvio:')
print(X_train_scaled[['TransactionAmt', 'TransactionAmt_log', 'hour_of_day', 'V257']]
      .agg(['mean', 'std']).round(4).to_string())

StandardScaler ajustado no treino e aplicado na validação.

Verificação (X_train_scaled) — deve ser ~0 média e ~1 desvio:
      TransactionAmt  TransactionAmt_log  hour_of_day    V257
mean          0.0000             -0.0000       0.0000 -0.0000
std           1.0000              1.0000       1.0000  1.0000


## 10. Salvando os Artefatos

Vamos salvar tudo que a Fase 4 (Modelagem) vai precisar:
- `X_train`, `X_val`, `y_train`, `y_val` — versão **sem** normalização (para XGBoost/LightGBM)
- `X_train_scaled`, `X_val_scaled` — versão **com** normalização (para Regressão Logística)
- `scaler.pkl` — o objeto StandardScaler treinado (para reaplicar em produção ou no teste)
- `feature_names.json` — lista de features na ordem certa

In [13]:
# Salvar splits em Parquet
X_train.to_parquet(os.path.join(DATA_DIR, 'X_train.parquet'), engine='pyarrow', index=False)
X_val.to_parquet(  os.path.join(DATA_DIR, 'X_val.parquet'),   engine='pyarrow', index=False)
X_train_scaled.to_parquet(os.path.join(DATA_DIR, 'X_train_scaled.parquet'), engine='pyarrow', index=False)
X_val_scaled.to_parquet(  os.path.join(DATA_DIR, 'X_val_scaled.parquet'),   engine='pyarrow', index=False)

y_train.to_frame().to_parquet(os.path.join(DATA_DIR, 'y_train.parquet'), engine='pyarrow', index=False)
y_val.to_frame().to_parquet(  os.path.join(DATA_DIR, 'y_val.parquet'),   engine='pyarrow', index=False)

# Salvar o scaler e a lista de features
with open(os.path.join(MODELS_DIR, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

with open(os.path.join(MODELS_DIR, 'feature_names.json'), 'w') as f:
    json.dump(feature_cols_final, f, indent=2)

print('Artefatos salvos:')
for fname in ['X_train.parquet','X_val.parquet','X_train_scaled.parquet',
              'X_val_scaled.parquet','y_train.parquet','y_val.parquet']:
    path = os.path.join(DATA_DIR, fname)
    print(f'  {fname}: {os.path.getsize(path)/1e6:.1f} MB')
print(f'  scaler.pkl e feature_names.json em {MODELS_DIR}')

Artefatos salvos:
  X_train.parquet: 52.1 MB
  X_val.parquet: 13.7 MB
  X_train_scaled.parquet: 53.5 MB
  X_val_scaled.parquet: 14.4 MB
  y_train.parquet: 0.0 MB
  y_val.parquet: 0.0 MB
  scaler.pkl e feature_names.json em /home/fernando-daniel-marcelino/workspace/projects/fraud_detection/models


## 11. Resumo do Pipeline

### O que o dataset final contém:

In [14]:
print('=== PIPELINE DE PRÉ-PROCESSAMENTO — RESUMO ===')
print(f'\nFeatures originais (após merge)       : 434 colunas')
print(f'Features removidas (ID + TransactionDT): {len(DROP_COLS)}')
print(f'Features novas (engenharia + missings) : {2 + 1 + len(new_indicator_cols)}')
print(f'Features finais                        : {X_train.shape[1]}')
print(f'\nConjunto de treino : {X_train.shape[0]:,} linhas  |  {y_train.sum():,} fraudes ({y_train.mean()*100:.2f}%)')
print(f'Conjunto de valid. : {X_val.shape[0]:,} linhas  |  {y_val.sum():,} fraudes ({y_val.mean()*100:.2f}%)')

print('\nFeatures criadas na engenharia:')
print('  hour_of_day, day_of_week, TransactionAmt_log')
print(f'  {len(new_indicator_cols)} indicadores de missing: {", ".join(new_indicator_cols)}')

print('\nPróxima fase: treinar Regressão Logística (baseline) e XGBoost/LightGBM (principal)')

=== PIPELINE DE PRÉ-PROCESSAMENTO — RESUMO ===

Features originais (após merge)       : 434 colunas
Features removidas (ID + TransactionDT): 2
Features novas (engenharia + missings) : 16
Features finais                        : 447

Conjunto de treino : 472,432 linhas  |  16,599 fraudes (3.51%)
Conjunto de valid. : 118,108 linhas  |  4,064 fraudes (3.44%)

Features criadas na engenharia:
  hour_of_day, day_of_week, TransactionAmt_log
  13 indicadores de missing: D6_is_nan, D7_is_nan, D8_is_nan, D9_is_nan, D12_is_nan, D13_is_nan, D14_is_nan, addr1_is_nan, addr2_is_nan, id_03_is_nan, id_04_is_nan, id_09_is_nan, id_10_is_nan

Próxima fase: treinar Regressão Logística (baseline) e XGBoost/LightGBM (principal)


## O que vem na Fase 4 — Modelagem

Com os dados prontos, vamos treinar dois modelos e comparar:

**Modelo 1 — Regressão Logística (baseline)**
- Usa `X_train_scaled` / `X_val_scaled`
- Parâmetro: `class_weight='balanced'`
- Por que serve como baseline? É simples, interpretável e rápido. Se o XGBoost não for muito melhor, há algo errado com a complexidade extra.

**Modelo 2 — XGBoost ou LightGBM (principal)**
- Usa `X_train` / `X_val` (sem normalização)
- Parâmetro: `scale_pos_weight` para desbalanceamento
- Por que melhor que Logística para este dataset? Captura relações não-lineares e interações entre features (ex: hora + valor + tipo de cartão combinados)

**Métricas que usaremos:**
- **AUC-ROC**: quão bem o modelo separa fraude de legítimo no geral
- **Average Precision (PR-AUC)**: mais informativa que ROC quando as classes são desbalanceadas
- **F1-score** no threshold ótimo: equilíbrio entre precision e recall